# 02 — Data Preprocessing & Feature Engineering

**职责**: 数据清洗 → 缺失值处理 → 特征构建 → 特征选择 → 输出处理后的特征矩阵

**输入**: `data/raw/train_data.csv`, `data/raw/test_data.csv`  
**输出**: `data/processed/train_processed.parquet`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from config import DATA_RAW, DATA_TEST, DATA_PROC, TARGET
from src.preprocessing import load_and_clean, fill_missing, build_feature_matrix
from src.feature_engineering import select_features, compute_ensemble_importance

## 1. 加载原始数据 & 基础清洗

In [2]:
df = load_and_clean(DATA_RAW)
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/src/preprocessing.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['startdate'] = pd.to_datetime(df['startdate'])


Shape: (375734, 251)
Missing values: 0


,index,lat,lon,startdate,contest-pevpr-sfc-gauss-14d__pevpr,nmme0-tmp2m-34w__cancm30,nmme0-tmp2m-34w__cancm40,nmme0-tmp2m-34w__ccsm30,nmme0-tmp2m-34w__ccsm40,nmme0-tmp2m-34w__cfsv20,...,wind-vwnd-925-2010-16,wind-vwnd-925-2010-17,wind-vwnd-925-2010-18,wind-vwnd-925-2010-19,wind-vwnd-925-2010-20,year,month,day,week,season
0,0,0.0,0.833333,2014-09-01,237.00,29.02,31.64,29.57,30.73,29.71,...,48.13,28.09,-13.50,11.90,4.58,2014,9,1,36,Fall
1,1,0.0,0.833333,2014-09-02,228.90,29.02,31.64,29.57,30.73,29.71,...,48.60,27.41,-23.77,15.44,3.42,2014,9,2,36,Fall
2,2,0.0,0.833333,2014-09-03,220.69,29.02,31.64,29.57,30.73,29.71,...,48.53,19.21,-33.16,15.11,4.82,2014,9,3,36,Fall
3,3,0.0,0.833333,2014-09-04,225.28,29.02,31.64,29.57,30.73,29.71,...,50.59,8.29,-37.22,18.24,9.74,2014,9,4,36,Fall
4,4,0.0,0.833333,2014-09-05,237.24,29.02,31.64,29.57,30.73,29.71,...,54.73,-2.58,-42.30,21.91,10.95,2014,9,5,36,Fall


## 2. 缺失值处理

检查填充后的数据质量。如果需要更精细的策略，可以修改 `src/preprocessing.py` 中的 `fill_missing()`。

In [3]:
missing_after = df.isnull().sum()
print(f"Remaining missing columns: {(missing_after > 0).sum()}")
if (missing_after > 0).any():
    print(missing_after[missing_after > 0].sort_values(ascending=False).head(10))

Remaining missing columns: 0


## 3. 特征选择

使用 Lasso + ElasticNet + RandomForest 集成方法筛选最优特征子集。

In [4]:
# TODO: 根据 EDA 结论调整 top_n（EDA 学习曲线的拐点）
X, y, selected_features = select_features(df, TARGET, top_n=30)
print(f"Selected {len(selected_features)} features")
print(f"X shape: {X.shape}, y shape: {y.shape}")

  Computing ensemble feature importance on 242 features...
  [1/3] Lasso …
  [2/3] ElasticNet …
  [3/3] RandomForest …
  Top-30 投票 ≥2/3: 25 个特征 (3/3=6, 2/3=19)
  Selected top 30 features (best: contest-slp-14d__slp)
Selected 25 features
X shape: (375734, 25), y shape: (375734,)


In [5]:
DATA_PROC.parent.mkdir(parents=True, exist_ok=True)

processed = X.copy()
processed[TARGET] = y.values
processed.to_csv(DATA_PROC, index=False)
print(f"Saved processed data to {DATA_PROC}")
print(f"Final shape: {processed.shape}")

Saved processed data to /Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/data/processed/train_processed.csv
Final shape: (375734, 26)


## 5. 划分训练集 & 测试集（80/20）

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(processed, test_size=0.2, random_state=42)

train_path = DATA_PROC.parent / 'train.csv'
test_path = DATA_PROC.parent / 'test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train: {train_df.shape} → {train_path}")
print(f"Test:  {test_df.shape} → {test_path}")